In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import keras 
import h5py as h5
from keras.models import Sequential
from keras.layers import Conv2D, Dense, MaxPool2D, Dropout, Flatten, Activation
from keras.optimizers import Adam
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import keras.backend as K
import tensorflow as tf

In [3]:
import os
print(os.listdir("../input"))
df_train = pd.read_csv('../input/train.csv')
X_train = df_train.iloc[:, 1:]
Y_train = df_train.iloc[:, 0]

['test.csv.zip', 'train.csv.zip', 'test.csv', 'train.csv']


In [4]:
df_test = pd.read_csv('../input/test.csv')
X_test = np.array(df_test)
X_test = X_test/255.0
np.save('test_input', X_test)
X_test = X_test.reshape((X_test.shape[0], 28, 28, 1))

In [4]:
from keras.models import Model
import math
from tensorflow.keras.models import load_model
#Lenet5 = load_model('models/SuperTinyLenet5_1convoKer.h5')
Lenet5 = load_model('models/SuperTinyLenet5.h5')
model = Lenet5

dummy_test = np.ones((28,28,1))
X_test[1] = np.ones((28,28,1))

layer_name = 'conv2d_10'

intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_test)

print((intermediate_output[1][0]))

layer_name = 'dense_17'
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_test)

print((intermediate_output[1]))
model.summary()

  1/875 [..............................] - ETA: 46s

2023-08-18 15:04:02.167779: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2023-08-18 15:04:02.167799: W tensorflow/stream_executor/cuda/cuda_driver.cc:263] failed call to cuInit: UNKNOWN ERROR (303)
2023-08-18 15:04:02.167811: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (rdsrv404): /proc/driver/nvidia/version does not exist
2023-08-18 15:04:02.167998: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


875/875 [==============================] - 0s 332us/step
[[1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        1.1072412]
 [1.4099426 0.9442534 4.418761  0.        

In [5]:
dense_w2 = Lenet5.layers[0].get_weights()[0]#.flatten(order='C')
dense_w2.shape

(5, 5, 1, 5)

In [6]:
# Assuming dense_w2 is your 4-dimensional array
newweight1 = dense_w2[:, :, :, :].reshape(-1, dense_w2.shape[3])
print(np.sum(dense_w2[:,:,:,4]))
print((dense_w2[:,:,:,1]))


1.0934755
[[[ 0.12231516]
  [ 0.3956783 ]
  [ 0.25372902]
  [ 0.25202024]
  [-0.07231694]]

 [[-0.07141761]
  [-0.12425912]
  [ 0.26498315]
  [ 0.09476443]
  [ 0.06650981]]

 [[-0.35749373]
  [-0.1607566 ]
  [ 0.04663221]
  [ 0.11312956]
  [ 0.25293663]]

 [[-0.21632108]
  [-0.2756716 ]
  [-0.10424771]
  [ 0.19096132]
  [ 0.22722453]]

 [[-0.53024846]
  [ 0.06306353]
  [-0.05334775]
  [ 0.26362312]
  [ 0.2925917 ]]]


In [7]:
dense_w2[1,0,:,0]

array([0.1175316], dtype=float32)

<H1> Building a model with only one kernel for each layer

In [8]:
model = Sequential()
model.add(Conv2D(filters=1, kernel_size=(5,5), padding='valid', input_shape=(28, 28, 1)))
model.add(Activation('relu'))
model.add(MaxPool2D(strides=2))
model.add(Conv2D(filters=1, kernel_size=(5,5), padding='valid'))
model.add(Activation('relu'))
model.add(MaxPool2D(strides=2))
model.add(Flatten())
model.add(Dense(100))
model.add(Activation('relu'))
model.add(Dense(50))
model.add(Activation('relu'))
model.add(Dense(10))
model.add(Activation('relu'))
model.build()
model.summary()
adam = Adam(learning_rate=5e-4)
model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer=adam)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 24, 24, 1)         26        
                                                                 
 activation (Activation)     (None, 24, 24, 1)         0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 12, 12, 1)        0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 1)           26        
                                                                 
 activation_1 (Activation)   (None, 8, 8, 1)           0         
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 4, 4, 1)          0         
 2D)                                                    

In [20]:
df_train = pd.read_csv('../input/train.csv')
X_train = df_train.iloc[:, 1:]
Y_train = df_train.iloc[:, 0]
X_train = np.array(X_train)
Y_train = np.array(Y_train)
# Normalize inputs
X_train = X_train / 255.0
#Train-Test Split
X_dev, X_val, Y_dev, Y_val = train_test_split(X_train, Y_train, test_size=0.03, shuffle=True, random_state=2019)
T_dev = pd.get_dummies(Y_dev).values
T_val = pd.get_dummies(Y_val).values
#Reshape the input 
X_dev = X_dev.reshape(X_dev.shape[0], 28, 28, 1)
X_val = X_val.reshape(X_val.shape[0], 28, 28, 1)

In [21]:
# Set a learning rate annealer
reduce_lr = ReduceLROnPlateau(monitor='val_loss', 
                                patience=3, 
                                verbose=1, 
                                factor=0.2, 
                                min_lr=1e-10)

In [22]:
# Data Augmentation
datagen = ImageDataGenerator(
            rotation_range=10, 
            width_shift_range=0.1, 
            height_shift_range=0.1, 
            zoom_range=0.1)
datagen.fit(X_dev)

In [23]:

model.fit(datagen.flow(X_dev, T_dev, batch_size=100), steps_per_epoch=len(X_dev)/100, 
                    epochs=20, validation_data=(X_val, T_val), callbacks=[reduce_lr])


Epoch 1/20
407/407 [==============================] - 3s 7ms/step - loss: 1.3971 - accuracy: 0.6161 - val_loss: 1.1399 - val_accuracy: 0.7611 - lr: 1.0000e-05
Epoch 2/20
407/407 [==============================] - 3s 7ms/step - loss: 1.4076 - accuracy: 0.6184 - val_loss: 1.1334 - val_accuracy: 0.7587 - lr: 1.0000e-05
Epoch 3/20
407/407 [==============================] - 3s 7ms/step - loss: 1.3940 - accuracy: 0.6187 - val_loss: 1.1359 - val_accuracy: 0.7571 - lr: 1.0000e-05
Epoch 4/20
407/407 [==============================] - 3s 8ms/step - loss: 1.4005 - accuracy: 0.6161 - val_loss: 1.1323 - val_accuracy: 0.7579 - lr: 1.0000e-05
Epoch 5/20
407/407 [==============================] - 3s 7ms/step - loss: 1.3981 - accuracy: 0.6210 - val_loss: 1.1150 - val_accuracy: 0.7603 - lr: 1.0000e-05
Epoch 6/20
407/407 [==============================] - 3s 7ms/step - loss: 1.4027 - accuracy: 0.6194 - val_loss: 1.1052 - val_accuracy: 0.7619 - lr: 1.0000e-05
Epoch 7/20
407/407 [==========================

In [24]:
score = model.evaluate(X_val, T_val, batch_size=1)

1260/1260 [==============================] - 1s 442us/step - loss: 1.0848 - accuracy: 0.7627


In [25]:
#Store the model
model.save('models/SuperTinyLenet5_AllKer1.h5')

In [5]:
from keras.models import Model
import math
from tensorflow.keras.models import load_model
#Lenet5 = load_model('models/SuperTinyLenet5_1convoKer.h5')
Lenet5 = load_model('models/SuperTinyLenet5_AllKer1.h5')
model = Lenet5
model.summary()
dummy_test = np.ones((28,28,1))
X_test[1] = np.ones((28,28,1))

layer_name = 'conv2d'

intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_test)

print((intermediate_output[1][0]))

layer_name = 'dense_2'
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_test)

print((intermediate_output[1]))


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 24, 24, 1)         26        
                                                                 
 activation (Activation)     (None, 24, 24, 1)         0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 12, 12, 1)        0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 1)           26        
                                                                 
 activation_1 (Activation)   (None, 8, 8, 1)           0         
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 4, 4, 1)          0         
 2D)                                                    

2023-08-21 13:52:17.294056: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2023-08-21 13:52:17.294068: W tensorflow/stream_executor/cuda/cuda_driver.cc:263] failed call to cuInit: UNKNOWN ERROR (303)
2023-08-21 13:52:17.294077: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (rdsrv404): /proc/driver/nvidia/version does not exist
2023-08-21 13:52:17.294628: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


875/875 [==============================] - 1s 303us/step
[[2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]
 [2.0384152]]
875/875 [==============================] - 0s 462us/step
[ 2.886335   -5.564783    1.7661021   1.9085015   0.16954198  0.42701584
  0.03657786 -0.31100845  1.9502566   0.61927044]


In [28]:
dense_w2 = Lenet5.layers[0].get_weights()[0]#.flatten(order='C')
dense_w2.shape

(5, 5, 1, 1)

<H1> Save all the weights to text file for CSIM on Vitis verification

In [30]:
convo_w0 = Lenet5.layers[0].get_weights()[0].flatten(order='C')
convo_b0 = Lenet5.layers[0].get_weights()[1].flatten(order='C')

convo_w1 = Lenet5.layers[3].get_weights()[0].flatten(order='C')
convo_b1 = Lenet5.layers[3].get_weights()[1].flatten(order='C')

dense_w0 = Lenet5.layers[7].get_weights()[0]#.flatten(order='C')
dense_b0 = Lenet5.layers[7].get_weights()[1].flatten(order='C')

dense_w1 = Lenet5.layers[9].get_weights()[0].flatten(order='C')
dense_b1 = Lenet5.layers[9].get_weights()[1].flatten(order='C')

dense_w2 = Lenet5.layers[11].get_weights()[0].flatten(order='C')
dense_b2 = Lenet5.layers[11].get_weights()[1].flatten(order='C')


# save to npy file

np.savetxt('1Filt_1Fet_STinyLenet5_convo2_w.txt', convo_w0)
np.savetxt('1Filt_1Fet_STinyLenet5_convo2_b.txt', convo_b0)

np.savetxt('1Filt_1Fet_STinyLenet5_convo3_w.txt', convo_w1)
np.savetxt('1Filt_1Fet_STinyLenet5_convo3_b.txt', convo_b1)

np.savetxt('1Filt_1Fet_STinyLenet5_Dense3_w.txt', dense_w0)
np.savetxt('1Filt_1Fet_STinyLenet5_Dense3_b.txt', dense_b0)

np.savetxt('1Filt_1Fet_STinyLenet5_Dense4_w.txt', dense_w1)
np.savetxt('1Filt_1Fet_STinyLenet5_Dense4_b.txt', dense_b1)

np.savetxt('1Filt_1Fet_STinyLenet5_Dense5_w.txt', dense_w2)
np.savetxt('1Filt_1Fet_STinyLenet5_Dense5_b.txt', dense_b2)


In [29]:
print(dense_w2)

[[[[ 0.24565808]]

  [[-0.3019829 ]]

  [[ 0.16596209]]

  [[ 0.2308325 ]]

  [[ 0.1917866 ]]]


 [[[-0.17224203]]

  [[ 0.1881944 ]]

  [[ 0.29362398]]

  [[ 0.21988562]]

  [[-0.28990304]]]


 [[[ 0.0704194 ]]

  [[ 0.30124053]]

  [[ 0.17642131]]

  [[-0.06260183]]

  [[ 0.11973343]]]


 [[[-0.08630358]]

  [[-0.18830258]]

  [[ 0.05629272]]

  [[ 0.03936193]]

  [[-0.14646442]]]


 [[[ 0.342049  ]]

  [[-0.11453977]]

  [[ 0.01955643]]

  [[ 0.38688514]]

  [[ 0.3496979 ]]]]
